# PLA-2D_ViT+trajectory

# Get Data

In [ ]:
import h5py

load_path='F:/NISTdataset/Active Mobile/AAPlantD1_2GHz_TX1_hpol_run4_pp.mat'
load_mat=h5py.File(load_path,'r')
len_sequence = 588  
IQdata=load_mat['IQdata'][3600:39600,0:len_sequence].view('complex')
IQdata_Range_m=(load_mat['IQdata_Range_m'][3:6,3600:39600]).transpose()
locations=120
records=300

# Hyper-parameters

In [ ]:
# training parameters
batch_size = 32                 # batch size
num_epoch = 100                 # the number of training epoch
patience = 5                    # If no improvement in 'patience' epochs, early stop
learning_rate = 0.00002         # learning rate

# model parameters
k=1                             # the number of request CIR
nf=2                            # the number of reference CIR
d=2                             # Eve is 2 positions behind Alice
input_channels=2                # the number of input channels
output_dim=2                    # binary classification

# Preprocess data 

In [ ]:
import numpy as np

interval_point=2  # interval point
def creatHimag_Alice(idx, k, nf):
    H=np.zeros((k+nf,len_sequence)).astype(complex)
    H=IQdata[idx-interval_point*nf:idx+k:interval_point]
    return H
def creatHimag_Eve(idx, k, nf, d):
    H=np.zeros((k+nf,len_sequence)).astype(complex)
    H[nf:k+nf,:]=IQdata[idx-d*records:idx+k-d*records]
    H[0:nf,:]=IQdata[idx-interval_point*nf:idx:interval_point]
    return H

In [ ]:
import gc

recs=records*d
num_dataall=locations*records-recs
dataall_HimagA=np.zeros((num_dataall, k+nf, len_sequence)).astype(complex)
dataall_HimagE=np.zeros((num_dataall, k+nf, len_sequence)).astype(complex)
for i in range(recs,locations*records):
    dataall_HimagA[i-recs,:,:]=creatHimag_Alice(i,k=k,nf=nf)
    dataall_HimagE[i-recs,:,:]=creatHimag_Eve(i,k=k,nf=nf,d=d)

dataall_Trajectory=np.zeros((num_dataall, 3*(nf+1))).astype(np.float32)
for i in range(recs,locations*records):
    dataall_Trajectory[i-recs,:]=(IQdata_Range_m[i-nf*interval_point:i+1:interval_point]).reshape(1,3*(nf+1))    

dataall_TW=np.concatenate((dataall_HimagA,dataall_HimagE),axis=0)
location=[i for i in range(2)]
all_lab=[val for val in location for i in range(int(num_dataall))]
print(dataall_TW.shape)

# training set:validation set:test set=6:2:2
train_index = np.zeros(int(len(dataall_TW)/5)*3, dtype=int) 
valid_index = [i for i in range(1, len(dataall_TW), 5)]
test_index = [i for i in range(3, len(dataall_TW), 5)]
for i in range(int(len(dataall_TW)/5)):
    train_index[i*3:(i+1)*3] = [0+5*i, 2+5*i, 4+5*i]

train_data=dataall_TW[train_index]
train_label=np.take(all_lab, train_index)
valid_data=dataall_TW[valid_index]
valid_label=np.take(all_lab, valid_index)
test_data=dataall_TW[test_index]
test_lab=np.take(all_lab, test_index)

dataall_TW1=np.concatenate((dataall_Trajectory,dataall_Trajectory),axis=0)
train_data1=dataall_TW1[train_index]
valid_data1=dataall_TW1[valid_index]
test_data1=dataall_TW1[test_index]

del dataall_HimagA, dataall_HimagE, dataall_TW, IQdata, all_lab, dataall_Trajectory, dataall_TW1, IQdata_Range_m
gc.collect()

# creat a image

In [ ]:
def convert_pic(h):
    height, width=h.shape
    img=np.zeros([input_channels,height,width]).astype(np.float32)
    img[0,:,:]=np.real(np.fft.fft(h))
    img[1,:,:]=np.imag(np.fft.fft(h))
    return img 

# Define Dataset

In [ ]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
    
class hDataset(Dataset):
    def __init__(self, dataset1, dataset2, y=None):
        super(hDataset).__init__()
        self.dataset1 = dataset1
        self.dataset2 = dataset2
        if y is not None:
            self.label = torch.LongTensor(y)
        else:
            self.label = None

    def __getitem__(self, idx):
        h1 = self.dataset1[idx]
        h2 = self.dataset2[idx]
        h1 = convert_pic(h1)
        if self.label is not None:
            return h1, h2, self.label[idx]
        else:
            return h1, h2

    def __len__(self):
        return len(self.dataset1)

# Prepare dataset and model

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from model import CombinedNetwork

# get dataset
train_set = hDataset(train_data, train_data1, train_label)
valid_set = hDataset(valid_data, valid_data1, valid_label)

# get dataloader
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'DEVICE: {device}')

net = CombinedNetwork(fimage_size = (nf+1, len_sequence), fpatch_size = nf+1, FCN_input_dim=3*(nf+1), inchannels=input_channels)
net.to(device)

criterion = nn.CrossEntropyLoss() 

params = [p for p in net.parameters() if p.requires_grad]

optimizer = optim.Adam(params, lr=learning_rate, weight_decay=1e-5)

# Training

In [ ]:
stale=0
best_acc = 0.0
_exp_name = "ViT-PLA"
train_losses=[]
train_accss=[]
valid_losses=[]
valid_accss=[]

for epoch in range(num_epoch):
    net.train()
    train_loss=[]
    train_accs=[]
# ---training---
    for batch in train_loader:    
        H1, H2, labels = batch
        H1 = H1.to(device)
        H2 = H2.to(device)
        labels = labels.to(device)
   
        optimizer.zero_grad()
        logits = net(H1, H2)
        loss =criterion(logits, labels)
        loss.backward()
        optimizer.step()
        
        acc=(logits.argmax(dim=-1) == labels).float().mean()
        
        # Record the loss and accuracy.
        train_loss.append(loss.item())
        train_accs.append(acc)
        
    train_loss = sum(train_loss) / len(train_loss)
    train_acc = sum(train_accs) / len(train_accs)
    train_losses.append(train_loss)
    train_accss.append(np.array(train_acc.cpu()))
    # Print the information.
    print(f"[ Train | {epoch + 1:03d}/{num_epoch:03d} ] loss = {train_loss:.5f}, acc = {train_acc:.5f}")

# ---validation---
    net.eval()
    valid_loss=[]
    valid_accs=[]

    for batch in valid_loader:
        # A batch consists of image data and corresponding labels.
        H1, H2, labels = batch
        H1 = H1.to(device)
        H2 = H2.to(device)
        labels = labels.to(device)

        # We don't need gradient in validation.
        # Using torch.no_grad() accelerates the forward process.
        with torch.no_grad():
            logits = net(H1, H2)

        # We can still compute the loss (but not the gradient).
        loss = criterion(logits, labels)

        # Compute the accuracy for current batch.
        acc = (logits.argmax(dim=-1) == labels).float().mean()

        # Record the loss and accuracy.
        valid_loss.append(loss.item())
        valid_accs.append(acc)

    # The average loss and accuracy for entire validation set is the average of the recorded values.
    valid_loss = sum(valid_loss) / len(valid_loss)
    valid_acc = sum(valid_accs) / len(valid_accs)
    valid_losses.append(valid_loss)
    valid_accss.append(np.array(valid_acc.cpu()))     

    if valid_acc > best_acc:
        print(f"[ Valid | {epoch + 1:03d}/{num_epoch:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f} -> best")
    else:
        print(f"[ Valid | {epoch + 1:03d}/{num_epoch:03d} ] loss = {valid_loss:.5f}, acc = {valid_acc:.5f}")


    # save models
    if valid_acc > best_acc:
        print(f"Best model found at epoch {epoch}, saving model")
        torch.save(net.state_dict(), f"{_exp_name}_best.ckpt") 
        best_acc = valid_acc
        stale = 0
    else:
        stale += 1
        if stale > patience:
            print(f"No improvment {patience} consecutive epochs, early stopping")
            break   

# Loss and Acc show

In [ ]:
import matplotlib.pyplot as plt

plt.figure(1)
plt.plot(train_losses, linestyle="--", label="train")
plt.plot(valid_losses, linestyle="-", label="valid")
plt.legend(loc="best")
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.show

plt.figure(2)
plt.plot(train_accss, linestyle="--", label="train")
plt.plot(valid_accss, linestyle="-", label="valid")
plt.legend(loc="best")
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.show

# Testing

In [ ]:
import time

test_set = hDataset(test_data, test_data1, test_lab)
test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)

model_best = CombinedNetwork(fimage_size = (nf+1, len_sequence), fpatch_size = nf+1, FCN_input_dim=3*(nf+1), inchannels=2).to(device)
model_best.load_state_dict(torch.load(f"{_exp_name}_best.ckpt"))
model_best.eval()
prediction = []

start_time = time.time()

with torch.no_grad():
    for data1, data2, _ in test_loader:
        test_pred = model_best(data1.to(device), data2.to(device))
        test_label = np.argmax(test_pred.cpu().data.numpy(), axis=1)
        prediction += test_label.squeeze().tolist()

end_time = time.time()
execution_time = end_time - start_time
print(f"Runtime: {execution_time} seconds")

accuracy=1 - np.count_nonzero(np.array(prediction)-np.array(test_lab))/len(prediction)
FAR=np.count_nonzero(np.array(prediction[0:int(len(prediction)/2)])-np.array(test_lab[0:int(len(prediction)/2)]))/len(prediction)*2
MDR=np.count_nonzero(np.array(prediction[int(len(prediction)/2):])-np.array(test_lab[int(len(prediction)/2):]))/len(prediction)*2
DER=1-accuracy
print(f"False alarm rate: {FAR}")
print(f"Miss detection rate: {MDR}")
print(f"Detection error rate: {DER}")
print(f"Accuracy: {accuracy}")

In [ ]:
error=np.array(prediction)-np.array(test_lab)

plt.figure(1)
plt.plot(error, linestyle="--")
plt.xlabel('Records')
plt.ylabel('Error')
plt.show